In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import os
import glob
import sys
import copy
import random
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from skopt.space import Integer, Categorical
from skopt import gp_minimize
from skopt.utils import use_named_args

In [20]:
data_path = r"path\Train_Test_Verification_Data.xlsx"

train_data = pd.read_excel(data_path, sheet_name="Train")
test_data = pd.read_excel(data_path, sheet_name="Test")


feature_train = train_data.iloc[:, :10].to_numpy(dtype=np.float32)
label_train = train_data.iloc[:, -1].to_numpy(dtype=np.float32)

feature_test = test_data.iloc[:, :10].to_numpy(dtype=np.float32)
label_test = test_data.iloc[:, -1].to_numpy(dtype=np.float32)

print("Train:", feature_train.shape, label_train.shape)
print("Test :", feature_test.shape, label_test.shape)

Train: (71520, 10) (71520,)
Test : (30652, 10) (30652,)


## Grid search of XGB

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score


RANDOM_SEED = 42

def evaluate_xgb(model, X, y):
    y_pred = model.predict(X)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2 = r2_score(y, y_pred)
    return rmse, r2, y_pred

n_estimators_list = list(range(5, 401, 5))  # 5, 10, ..., 400
max_depth_list = list(range(1, 11))          # 1, 2, ..., 8


results = []

best_rmse = float("inf")
best_params = None
best_model = None

trial = 0
total_trials = len(n_estimators_list) * len(max_depth_list)

In [ ]:
for n_estimators in n_estimators_list:
    for max_depth in max_depth_list:
        trial += 1

        model = XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            objective="reg:squarederror",
            random_state=RANDOM_SEED,
            n_jobs=-1
        )

        model.fit(feature_train, label_train)

        val_rmse, val_r2, _ = evaluate_xgb(
            model,
            feature_test,
            label_test
        )

        results.append({
            "Trial": trial,
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "Validation_RMSE": val_rmse,
            "Validation_R2": val_r2
        })

        if val_rmse < best_rmse:
            best_rmse = val_rmse
            best_params = {
                "n_estimators": n_estimators,
                "max_depth": max_depth
            }
            best_model = model

        print(
            f"[{trial}/{total_trials}] "
            f"n_estimators={n_estimators}, "
            f"max_depth={max_depth}, "
            f"val_RMSE={val_rmse:.6f}, "
            f"val_R2={val_r2:.6f}"
        )



grid_history = pd.DataFrame(results)
grid_history["Best_RMSE_so_far"] = grid_history["Validation_RMSE"].cummin()

[1/800] n_estimators=5, max_depth=1, val_RMSE=13.357762, val_R2=0.847811
[2/800] n_estimators=5, max_depth=2, val_RMSE=10.804164, val_R2=0.900437
[3/800] n_estimators=5, max_depth=3, val_RMSE=9.408040, val_R2=0.924505
[4/800] n_estimators=5, max_depth=4, val_RMSE=8.605031, val_R2=0.936843
[5/800] n_estimators=5, max_depth=5, val_RMSE=8.169960, val_R2=0.943068
[6/800] n_estimators=5, max_depth=6, val_RMSE=7.565875, val_R2=0.951176
[7/800] n_estimators=5, max_depth=7, val_RMSE=7.317251, val_R2=0.954332
[8/800] n_estimators=5, max_depth=8, val_RMSE=7.431643, val_R2=0.952893
[9/800] n_estimators=5, max_depth=9, val_RMSE=7.253935, val_R2=0.955119
[10/800] n_estimators=5, max_depth=10, val_RMSE=7.373196, val_R2=0.953631
[11/800] n_estimators=10, max_depth=1, val_RMSE=11.018058, val_R2=0.896455
[12/800] n_estimators=10, max_depth=2, val_RMSE=8.468599, val_R2=0.938830
[13/800] n_estimators=10, max_depth=3, val_RMSE=6.992506, val_R2=0.958296
[14/800] n_estimators=10, max_depth=4, val_RMSE=6.093

Exception ignored on calling ctypes callback function: <bound method DataIter._next_wrapper of <xgboost.data.SingleBatchInternalIter object at 0x0000023E67C39730>>
Traceback (most recent call last):
  File "D:\anaconda\anaconda\lib\site-packages\xgboost\core.py", line 582, in _next_wrapper
    def _next_wrapper(self, this: None) -> int:  # pylint: disable=unused-argument
KeyboardInterrupt: 


## Bayesian optimization of XGB

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
from skopt import gp_minimize
from skopt.space import Integer
from skopt.utils import use_named_args

RANDOM_SEED = 42
N_CALLS = 500
N_INITIAL_POINTS = 50

trial_count = 0
best_rmse_so_far = float("inf")
best_params_so_far = None
bo_history = []
bo_start_time = time.time()

In [ ]:
def evaluate_xgb(model, X, y):
    y_pred = model.predict(X)
    mse = mean_squared_error(y, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y, y_pred)
    return rmse, r2, y_pred



space = [
    Integer(1, 80, name="n_estimators_index"), 
    Integer(1, 10, name="max_depth")
]

@use_named_args(space)
def objective(n_estimators_index, max_depth): 
    global trial_count, best_rmse_so_far, best_params_so_far, bo_history

    trial_count += 1
    trial_start_time = time.time()

    n_estimators = int(n_estimators_index) * 5

    model = XGBRegressor(
        n_estimators=n_estimators,  
        max_depth=int(max_depth),
        objective="reg:squarederror",
        random_state=RANDOM_SEED,
        n_jobs=-1
    )

    model.fit(feature_train, label_train)

    val_rmse, val_r2, _ = evaluate_xgb(
        model,
        feature_test,
        label_test
    )


    if val_rmse < best_rmse_so_far:
        best_rmse_so_far = val_rmse
        best_params_so_far = {
            "n_estimators": n_estimators,
            "max_depth": int(max_depth)
        }

    trial_time = time.time() - trial_start_time
    elapsed_time = time.time() - bo_start_time
    avg_time = elapsed_time / trial_count
    eta_time = avg_time * (N_CALLS - trial_count)

    bo_history.append({
        "Trial": trial_count,
        "n_estimators_index": int(n_estimators_index),  
        "n_estimators": n_estimators,                  
        "max_depth": int(max_depth),
        "Validation_RMSE": val_rmse,
        "Validation_R2": val_r2,
        "Best_RMSE_so_far": best_rmse_so_far,
        "Best_n_estimators_so_far": best_params_so_far["n_estimators"],
        "Best_max_depth_so_far": best_params_so_far["max_depth"],
        "Trial_time_s": trial_time,
        "Elapsed_time_min": elapsed_time / 60,
        "ETA_min": eta_time / 60
    })

    history_running = pd.DataFrame(bo_history)

 
    print(
        f"Trial {trial_count}/{N_CALLS} | "
        f"n_estimators={n_estimators}, "
        f"max_depth={max_depth}, "
        f"val_RMSE={val_rmse:.6f}, "
        f"val_R2={val_r2:.6f} | "
        f"Best_RMSE_so_far={best_rmse_so_far:.6f}, "
        f"Best_params_so_far={best_params_so_far} | "
        f"Trial_time={trial_time:.1f}s, "
        f"Elapsed={elapsed_time/60:.1f} min, "
        f"ETA={eta_time/60:.1f} min"
    )

    return val_rmse

In [6]:
result = gp_minimize(
    func=objective,
    dimensions=space,
    n_calls=N_CALLS,
    n_initial_points=N_INITIAL_POINTS,
    acq_func="EI",      # acquisition function: Expected Improvement
    random_state=RANDOM_SEED
)


best_params = {
    "n_estimators": int(result.x[0]) * 5,
    "max_depth": int(result.x[1])
}

print("\nBest validation RMSE:", result.fun)
print("Best hyperparameters:")
for k, v in best_params.items():
    print(f"{k}: {v}")

Trial 1/500 | n_estimators=320, max_depth=3, val_RMSE=4.129051, val_R2=0.985458 | Best_RMSE_so_far=4.129051, Best_params_so_far={'n_estimators': 320, 'max_depth': 3} | Trial_time=0.4s, Elapsed=0.0 min, ETA=3.4 min
Trial 2/500 | n_estimators=315, max_depth=6, val_RMSE=4.371455, val_R2=0.983701 | Best_RMSE_so_far=4.129051, Best_params_so_far={'n_estimators': 320, 'max_depth': 3} | Trial_time=0.7s, Elapsed=0.0 min, ETA=4.5 min
Trial 3/500 | n_estimators=180, max_depth=2, val_RMSE=5.373818, val_R2=0.975369 | Best_RMSE_so_far=4.129051, Best_params_so_far={'n_estimators': 320, 'max_depth': 3} | Trial_time=0.2s, Elapsed=0.0 min, ETA=3.5 min
Trial 4/500 | n_estimators=185, max_depth=4, val_RMSE=4.984046, val_R2=0.978812 | Best_RMSE_so_far=4.129051, Best_params_so_far={'n_estimators': 320, 'max_depth': 3} | Trial_time=0.3s, Elapsed=0.0 min, ETA=3.3 min


KeyboardInterrupt: 

## Bayesian optimization of DNN

In [9]:
class ExcelDataset(Dataset):
    def __init__(self):
        self.feature = feature_train  
        self.label=label_train  
        self.x = torch.from_numpy(self.feature).float()
        self.y = torch.from_numpy(self.label).float()
        
    def __len__(self):
        return len(self.y)
    def __getitem__(self, index):
        x=self.x[index]
        y=self.y[index]
        return {'x':x,'y':y}

dataset=ExcelDataset()
dataloader=DataLoader(dataset=dataset, batch_size=256,shuffle=True,num_workers=0,drop_last = True)

In [10]:
import torch.nn.init as init
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

class BP(nn.Module):
    def __init__(self, num_layers,hidden_size):
        super(BP,self).__init__()
        self.Liner1 = nn.Linear(10,hidden_size)
        self.queue= [nn.Sequential(nn.Linear(hidden_size, hidden_size), nn.LeakyReLU(0.1)) for _ in range(num_layers - 1)]
        self.model=nn.Sequential(*self.queue)
        self.Liner2 = nn.Linear(hidden_size,1)
        
        
    def forward(self,input):
        output = self.Liner1(input)
        output=self.model(output)
        output = self.Liner2(output)
        return output
MSEloss = nn.MSELoss() 

In [11]:
N_CALLS = 80
bo_history = []          
bo_start_time = time.time()

In [15]:
def train_one_model(num_layers, hidden_size, n_epochs, patience, min_delta):
    set_seed(42)

    bp = BP(num_layers, hidden_size)
    optimizer = torch.optim.Adam(bp.parameters(), lr=0.01)

    best_val_loss = float("inf")
    best_model = None
    wait = 0
    best_epoch = 0 

    for epoch in range(n_epochs):
        bp.train()

        for value in dataloader:
            x = value["x"]
            y = value["y"]

            y_train = bp(x).squeeze(1)
            loss = MSEloss(y, y_train)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        bp.eval()
        with torch.no_grad():
            _, val_loss = test(bp, feature_test, label_test)

        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_model = copy.deepcopy(bp.state_dict())
            wait = 0
            best_epoch = epoch + 1
        else:
            wait += 1

        if epoch > 10 and wait >= patience:
            break

    if best_model is not None:
        bp.load_state_dict(best_model)

    return best_val_loss, best_epoch 


space = [
    Integer(2, 10, name="num_layers"),
    Categorical([2, 4, 6, 8, 10, 12, 16, 20, 24, 28, 32, 48, 64, 96,128], name="hidden_size")
]

@use_named_args(space)
def objective(num_layers, hidden_size):
    global bo_history

    trial_id = len(bo_history) + 1 
    trial_start_time = time.time() 

    val_loss, best_epoch = train_one_model( 
        num_layers=int(num_layers),
        hidden_size=int(hidden_size),
        n_epochs=50,
        patience=5,
        min_delta=1e-1
    )

    
    trial_time = time.time() - trial_start_time
    elapsed_time = time.time() - bo_start_time

    avg_time = elapsed_time / trial_id
    eta_time = avg_time * (N_CALLS - trial_id)

 
    bo_history.append({
        "Trial": trial_id,
        "num_layers": int(num_layers),
        "hidden_size": int(hidden_size),
        "Best_epoch": best_epoch,
        "Validation_loss": val_loss,
        "Trial_time_s": trial_time,
        "Elapsed_time_min": elapsed_time / 60,
        "ETA_min": eta_time / 60
    })

    history_df = pd.DataFrame(bo_history)
    history_df["Best_loss_so_far"] = np.minimum.accumulate(history_df["Validation_loss"])



    print(
        f"\nTrial {trial_id}/{N_CALLS} | "
        f"num_layers={num_layers}, hidden_size={hidden_size}, "
        f"best_epoch={best_epoch}, val_loss={val_loss:.6f} | "
        f"Trial time={trial_time:.1f}s | "
        f"Elapsed={elapsed_time/60:.1f} min | "
        f"ETA={eta_time/60:.1f} min"
    )

    return val_loss

In [16]:
def test(model, feature_test, label_test):
    predict_list=[]
    model.eval()
    for test_feature in feature_test:
        # print(f'test_feature:{test_feature}')
        test_feature = torch.from_numpy(test_feature).float()
        test_feature=test_feature.unsqueeze(0)
        #test_feature=test_feature.cuda()
        predict=model(test_feature)
        predict=predict.squeeze(1)
        # print(f'predict:{predict}')
        predict=predict.squeeze(0)
        predict_list.append(predict.detach().numpy())
 
    predict_list=np.array(predict_list)
    loss = MSEloss(torch.from_numpy(predict_list),torch.from_numpy(label_test)).item()
    #print("[R2:{}][loss:{}]".format(r2_score(label_test, predict_list), loss))
    return predict_list, loss

In [17]:
result = gp_minimize(
    func=objective,
    dimensions=space,
    n_calls=N_CALLS,          
    n_initial_points=10,
    acq_func="EI",
    random_state=42
)

print("Best validation loss:", result.fun)
print("Best num_layers:", result.x[0])
print("Best hidden_size:", result.x[1])


Trial 1/80 | num_layers=8, hidden_size=6, best_epoch=14, val_loss=18.908821 | Trial time=214.1s | Elapsed=4.2 min | ETA=334.3 min


KeyboardInterrupt: 